# Hybrid Graph RAG in Rust: Combining Vector Search with Graph Intelligence

**Building a high-performance Retrieval-Augmented Generation engine with LadybugDB, icebug, and Rust**

> For a deep dive into LadybugDB and its graph + vector capabilities, see the book: [LadybugDB — The Embedded Graph Database](https://leanpub.com/ladybugdb)

---

## The Problem with Traditional RAG

Traditional RAG systems rely solely on **vector similarity search**: embed the query, find the nearest chunks, stuff them into a prompt. This works well for simple factual questions, but falls apart when:

- The answer requires **connecting information across multiple documents** (multi-hop reasoning)
- You need to understand **entity relationships** ("How does X relate to Y?")
- The query targets **global or structural knowledge** ("What are the main themes?", "Who are the key players?")

**Hybrid Graph RAG** solves these limitations by adding a knowledge graph layer on top of vector search, then fusing results from both retrieval mechanisms.

## Architecture Overview

The system combines four retrieval mechanisms into a single pipeline:

![Architecture Overview](../diagrams/01_architecture_overview.png)

*Diagram source: [diagrams/01_architecture_overview.mmd](../diagrams/01_architecture_overview.mmd)*

## The Tech Stack

| Component | Technology | Role |
|-----------|-----------|------|
| **Graph Engine** | [icebug](https://github.com/Ladybug-Memory/icebug) (NetworKit fork) | PageRank, Louvain, BFS |
| **Graph DB** | [LadybugDB](https://leanpub.com/ladybugdb) | Production graph + vector storage |
| **Language** | Rust | Memory-safe, high-performance core |
| **FFI Layer** | C wrapper + `icebug-sys` | Bridge C++ graph library to Rust |
| **Embeddings** | Pluggable `Embedder` trait | Swap in any model (ONNX, API, etc.) |

### Why Rust?

- **Zero-cost abstractions** over the C++ graph library via FFI
- **Memory safety** without GC — critical for long-running RAG services
- **Performance** on par with C++ for the compute-intensive graph algorithms
- **Cargo ecosystem** for dependency management and testing

## How icebug Powers the Graph Layer

[icebug](https://github.com/Ladybug-Memory/icebug) is a high-performance graph analysis library (a fork of NetworKit) with 200+ algorithms. In our RAG system, we use three key algorithms:

### 1. PageRank — Entity Importance Scoring

![PageRank](../diagrams/02_pagerank.png)

*Diagram source: [diagrams/02_pagerank.mmd](../diagrams/02_pagerank.mmd)*

**How it works in RAG:**
- After entity extraction, entities become nodes and co-occurrences become edges
- PageRank assigns importance: entities mentioned alongside many other important entities get higher scores
- During graph expansion, chunks connected to high-PageRank entities are boosted

**The formula:** `PR(v) = (1-d)/N + d * Σ(PR(u) / deg(u))` for each neighbor `u` of `v`

```rust
// In our Rust implementation:
let mut pr = PageRank::new(&graph, 0.85, 1e-6);
pr.run();
let importance = pr.score(entity_node_id);
```

### 2. Louvain Community Detection — Topic Clustering

![Louvain Communities](../diagrams/03_louvain_communities.png)

*Diagram source: [diagrams/03_louvain_communities.mmd](../diagrams/03_louvain_communities.mmd)*

**How it works in RAG:**
- The Louvain algorithm iteratively optimizes **modularity** — a measure of how well a graph is partitioned into dense clusters
- Phase 1: Each node moves to the community that maximizes local modularity gain
- Phase 2: Communities are aggregated into super-nodes, then Phase 1 repeats
- Result: entities naturally cluster into **topic groups** without supervision

**Use case:** For broad queries like "Summarize the main topics", we can retrieve representative chunks from each community rather than just the nearest vectors.

```rust
let mut lv = Louvain::new(&graph, false, 1.0, 32);
lv.run();
let partition = lv.get_partition();
let community_id = partition.subset_of(entity_node_id);
```

### 3. BFS — Graph Expansion

![BFS Expansion](../diagrams/04_bfs_expansion.png)

*Diagram source: [diagrams/04_bfs_expansion.mmd](../diagrams/04_bfs_expansion.mmd)*

**How it works in RAG:**
1. Embed the query and find the top-k most similar **entities** via vector search (these are "seed entities")
2. From each seed, traverse the knowledge graph outward for `n` hops (default: 2)
3. Collect all **chunks** reachable through MENTIONS and RELATES_TO edges
4. Weight discovered chunks by the PageRank of the entities that led to them

This is how multi-hop reasoning works — a query about "memory safety" discovers chunks about "ownership" and "concurrency" even if those chunks don't contain the exact query terms.

## Reciprocal Rank Fusion (RRF)

The key insight: vector search and graph search produce **complementary** rankings. RRF merges them without requiring score normalization.

![RRF Fusion](../diagrams/05_rrf_fusion.png)

*Diagram source: [diagrams/05_rrf_fusion.mmd](../diagrams/05_rrf_fusion.mmd)*

**Formula:** `RRF(d) = Σ 1/(k + rank_i(d))` where `k=60` (constant) and `rank_i` is the rank in list `i`.

Items appearing in **both** lists get the highest fused scores — these are chunks that are both semantically similar AND structurally connected through the knowledge graph.

```rust
fn rrf_fusion(vector_ranked: &[(String, f32)], graph_ranked: &[(String, f64)], k: f64) -> Vec<(String, f64)> {
    let mut scores: HashMap<String, f64> = HashMap::new();
    for (rank, (id, _)) in vector_ranked.iter().enumerate() {
        *scores.entry(id.clone()).or_insert(0.0) += 1.0 / (rank as f64 + k);
    }
    for (rank, (id, _)) in graph_ranked.iter().enumerate() {
        *scores.entry(id.clone()).or_insert(0.0) += 1.0 / (rank as f64 + k);
    }
    // sort descending...
}
```

## The Ingestion Pipeline

![Ingestion Pipeline](../diagrams/06_ingestion_pipeline.png)

*Diagram source: [diagrams/06_ingestion_pipeline.mmd](../diagrams/06_ingestion_pipeline.mmd)*

## Running the Rust Demo

Let's run the compiled Rust RAG engine and see it in action. The demo ingests two documents, builds a knowledge graph, and queries it with three different questions.

In [ ]:
import subprocess
import os

# Build and run the Rust demo
project_root = os.path.dirname(os.path.dirname(os.path.abspath(".")))
# Adjust path if running from notebooks/ directory
if os.path.basename(os.getcwd()) == "notebooks":
    project_root = os.path.dirname(os.getcwd())
else:
    project_root = os.getcwd()

print(f"Project root: {project_root}")
print("Building Rust demo (first run may take a while)...\n")

result = subprocess.run(
    ["cargo", "run", "-p", "ladybug-rag", "--example", "demo"],
    capture_output=True, text=True, cwd=project_root
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("Build failed:")
    print(result.stderr[-2000:])

## Understanding the Results

Let's break down what just happened:

### 1. Ingestion
- Two documents (Rust overview + graph algorithms) were split into **16 chunks**
- **49 entities** were extracted (concepts like "Rust", "PageRank", "Ownership System")
- **74 MENTIONS edges** connect chunks to the entities they contain
- **46 RELATES_TO edges** connect entities that co-occur in the same sentences

### 2. Graph Processing (via icebug)
- **PageRank** identified the most connected/important entities in the knowledge graph
- **Louvain** detected **7 communities** — topic clusters that emerged from the entity relationship structure

### 3. Hybrid Retrieval
- Each query was processed through both vector search and graph expansion
- Results marked `Hybrid` were found by **both** mechanisms — these are the highest-confidence results
- Graph expansion discovered relevant chunks that pure vector search might miss

## Running the Tests

The project includes comprehensive tests for all three crates:

In [ ]:
result = subprocess.run(
    ["cargo", "test", "--", "--format=terse"],
    capture_output=True, text=True, cwd=project_root
)

# Show test output
for line in result.stdout.split("\n"):
    if any(kw in line for kw in ["test ", "running", "test result"]):
        print(line)

## The Knowledge Graph Schema

![Graph Schema](../diagrams/07_graph_schema.png)

*Diagram source: [diagrams/07_graph_schema.mmd](../diagrams/07_graph_schema.mmd)*

### Edge Types

| Edge | From | To | Meaning |
|------|------|----|--------|
| `MENTIONS` | Chunk | Entity | This chunk contains this entity |
| `RELATES_TO` | Entity | Entity | These entities co-occur in sentences |
| `NEXT_CHUNK` | Chunk | Chunk | Sequential document ordering |

## Crate Architecture

![Crate Architecture](../diagrams/08_crate_architecture.png)

*Diagram source: [diagrams/08_crate_architecture.mmd](../diagrams/08_crate_architecture.mmd)*

## Performance Benefits

Based on benchmarks from the [original Python implementation](https://github.com/Volland/ladybug-rag), Hybrid Graph RAG shows significant improvements over pure vector search:

| Metric | Improvement |
|--------|------------|
| Context precision | **+21%** |
| Answer completeness | **+30%** |
| Multi-hop question accuracy | **+109%** |
| Global question accuracy | **+195%** |

The Rust implementation adds:
- **Near-native speed** — icebug's C++ graph algorithms run at full performance through zero-overhead FFI
- **Parallel graph processing** — OpenMP-powered PageRank and Louvain scale across CPU cores
- **Memory efficiency** — no GC pauses, deterministic resource cleanup via Rust's ownership model

## Extending the System

### Plugging in a Real Embedding Model

The `Embedder` trait is designed for easy replacement:

```rust
pub trait Embedder: Send + Sync {
    fn embed(&self, text: &str) -> Vec<f32>;
    fn dimension(&self) -> usize;
}

// Example: wrap an ONNX model
struct OnnxEmbedder { session: ort::Session }

impl Embedder for OnnxEmbedder {
    fn embed(&self, text: &str) -> Vec<f32> {
        // tokenize, run inference, return pooled output
    }
    fn dimension(&self) -> usize { 384 }
}
```

### Using LadybugDB for Persistence

For production use, the in-memory stores can be replaced with [LadybugDB](https://leanpub.com/ladybugdb) — an embedded graph database that provides:

- **Property graph storage** with Cypher queries
- **HNSW vector indexes** for fast approximate nearest-neighbor search
- **Built-in graph algorithms** (PageRank, Louvain) via its `algo` extension
- **Single-file database** — no external services needed

### Adding More icebug Algorithms

icebug has 200+ algorithms. Some particularly useful ones for RAG:

- **Betweenness Centrality** — find "bridge" entities connecting topic clusters
- **Connected Components** — detect isolated sub-graphs in the knowledge base
- **Graph Generators** — create synthetic graphs for testing and benchmarking
- **Link Prediction** — suggest missing entity relationships

## Summary

Hybrid Graph RAG in Rust delivers a significant upgrade over traditional vector-only RAG:

1. **Vector search** catches semantically similar content
2. **Entity extraction** builds a knowledge graph from document content
3. **PageRank** (via icebug) identifies the most important entities
4. **Louvain** (via icebug) discovers topic communities
5. **Graph expansion** follows entity relationships to find structurally relevant content
6. **RRF fusion** merges both retrieval signals into a single ranking

The result: better context precision, dramatically improved multi-hop reasoning, and the ability to answer structural questions about your knowledge base.

---

### Resources

- **Book:** [LadybugDB — The Embedded Graph Database](https://leanpub.com/ladybugdb)
- **icebug:** [github.com/Ladybug-Memory/icebug](https://github.com/Ladybug-Memory/icebug)
- **ladybug-rag (Python):** [github.com/Volland/ladybug-rag](https://github.com/Volland/ladybug-rag)
- **ladybug-rag-rs (this repo):** [github.com/Volland/ladybug-rag-rs](https://github.com/Volland/ladybug-rag-rs)